# IICS Project Mapping & Transformation Comparator

This notebook compares two IICS/IDMC projects using REST APIs.

It will:

- Login to IICS.
- Retrieve assets from Project A and Project B.
- Identify common assets, assets only in A, and assets only in B.
- Match common mapping-oriented assets by relative path and type.
- Export matching assets from both projects using Platform REST API v3.
- Download and unzip the export packages.
- Locate exported JSON definitions.
- Normalize volatile IDs, timestamps, audit metadata, and environment-specific values.
- Compare transformations, fields/ports, expressions, conditions, parameters, and connections.
- Produce a full recursive JSON structural diff as a fallback.
- Export CSV and Excel comparison reports.

The parser is deliberately tolerant of IICS export JSON differences between releases.


## 1. Install dependencies

In [ ]:
# Uncomment if needed
# %pip install requests pandas openpyxl


## 2. Configuration

In [ ]:
import os

LOGIN_URL = os.getenv(
    "IICS_LOGIN_URL",
    "https://dm-us.informaticacloud.com/ma/api/v2/user/login"
)

USERNAME = os.getenv("IICS_USERNAME", "")
PASSWORD = os.getenv("IICS_PASSWORD", "")

# Update these
PROJECT_A = "DEV/Claims"
PROJECT_B = "PROD/Claims"

# Match by path below each project + asset type.
MATCH_BY_RELATIVE_PATH = True

# Common mapping-oriented IICS types.
# Set to None to deep-compare all common exportable asset types.
COMPARE_ASSET_TYPES = {
    "MAPPING",
    "DTEMPLATE",
    "MTT",
    "MAPPLET",
    "TASKFLOW",
    "WORKFLOW"
}

INCLUDE_DEPENDENCIES = False
EXPORT_BATCH_SIZE = 20
EXPORT_POLL_SECONDS = 2
EXPORT_TIMEOUT_SECONDS = 1800

OUTPUT_DIR = "iics_project_comparison"

print("Project A:", PROJECT_A)
print("Project B:", PROJECT_B)


## 3. Imports and helper functions

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import time
import zipfile
from datetime import datetime
from pathlib import Path, PurePosixPath

import pandas as pd
import requests
from IPython.display import display

REQUEST_TIMEOUT = 90


def safe_json(response):
    try:
        return response.json()
    except Exception as exc:
        raise RuntimeError(
            f"Expected JSON from {response.url}; "
            f"HTTP {response.status_code}: {response.text[:1500]}"
        ) from exc


def check_response(response):
    if not response.ok:
        raise RuntimeError(
            f"HTTP {response.status_code} calling "
            f"{response.request.method} {response.url}\n"
            f"{response.text[:3000]}"
        )
    return response


def normalize_project_path(path):
    path = str(path or "").strip().replace("\\", "/")
    path = re.sub(r"/+", "/", path)
    return path.strip("/")


def normalize_asset_type(value):
    return str(value or "").strip().upper()


def first_nonempty(obj, keys, default=None):
    if not isinstance(obj, dict):
        return default
    for key in keys:
        value = obj.get(key)
        if value not in (None, "", [], {}):
            return value
    return default


def compact_json(value):
    return json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        default=str
    )


## 4. IICS REST client

In [ ]:
class IICSClient:
    def __init__(self, login_url, username, password):
        self.login_url = login_url.rstrip("/")
        self.username = username
        self.password = password
        self.session_id = None
        self.server_url = None
        self.base_api_url = None
        self.org_id = None

    def login(self):
        payload = {
            "@type": "login",
            "username": self.username,
            "password": self.password
        }

        r = requests.post(
            self.login_url,
            headers={
                "Accept": "application/json",
                "Content-Type": "application/json"
            },
            json=payload,
            timeout=REQUEST_TIMEOUT
        )
        check_response(r)
        data = safe_json(r)

        self.session_id = data.get("icSessionId")
        self.server_url = (data.get("serverUrl") or "").rstrip("/")
        self.base_api_url = (
            data.get("baseApiUrl")
            or data.get("baseAPIUrl")
            or self.server_url
        ).rstrip("/")
        self.org_id = data.get("orgId") or data.get("organizationId")

        if not self.session_id:
            raise RuntimeError("Login response did not return icSessionId.")
        if not self.base_api_url:
            raise RuntimeError(
                "Login response did not return baseApiUrl/serverUrl."
            )

        return data

    @property
    def headers(self):
        if not self.session_id:
            raise RuntimeError("Call login() first.")

        # Different IICS resources/releases use one of these headers.
        return {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "INFA-SESSION-ID": self.session_id,
            "icSessionId": self.session_id
        }

    def get(self, path, params=None, stream=False):
        url = (
            path if str(path).startswith("http")
            else f"{self.base_api_url}/{str(path).lstrip('/')}"
        )
        r = requests.get(
            url,
            headers=self.headers,
            params=params,
            timeout=REQUEST_TIMEOUT,
            stream=stream
        )
        return check_response(r)

    def post(self, path, payload):
        url = (
            path if str(path).startswith("http")
            else f"{self.base_api_url}/{str(path).lstrip('/')}"
        )
        r = requests.post(
            url,
            headers=self.headers,
            json=payload,
            timeout=REQUEST_TIMEOUT
        )
        return check_response(r)

    @staticmethod
    def _extract_objects(payload):
        if isinstance(payload, list):
            return payload
        if not isinstance(payload, dict):
            return []

        for key in ("objects", "entries", "items", "results"):
            value = payload.get(key)
            if isinstance(value, list):
                return value

        if "id" in payload and (
            "name" in payload or
            "path" in payload or
            "location" in payload
        ):
            return [payload]

        return []

    @staticmethod
    def object_path(obj):
        path = first_nonempty(
            obj,
            ["path", "fullPath", "objectPath", "location", "sourcePath"],
            default=""
        )
        name = first_nonempty(
            obj,
            ["name", "objectName", "assetName"],
            default=""
        )

        path = str(path or "").replace("\\", "/").strip()

        if path and name and obj.get("location") == path:
            clean = path.rstrip("/")
            if not clean.endswith("/" + str(name)):
                path = clean + "/" + str(name)

        if not path:
            path = str(name or "")

        return "/" + normalize_project_path(path)

    @staticmethod
    def path_is_under_project(full_path, project_path):
        fp = normalize_project_path(full_path).casefold()
        pp = normalize_project_path(project_path).casefold()
        return fp == pp or fp.startswith(pp + "/")

    def list_objects(self, project_path):
        project_path = normalize_project_path(project_path)
        endpoint = "/public/core/v3/objects"

        # POD/release query syntax can differ, so try common forms and
        # always enforce project scoping locally.
        attempts = [
            {"q": f"location=='{project_path}'"},
            {"q": f"location=='/{project_path}'"},
            {}
        ]

        last_error = None

        for params in attempts:
            try:
                payload = safe_json(self.get(endpoint, params=params))
                rows = self._extract_objects(payload)

                scoped = [
                    obj for obj in rows
                    if self.path_is_under_project(
                        self.object_path(obj),
                        project_path
                    )
                ]

                if scoped:
                    return scoped

                if not rows:
                    return []

            except Exception as exc:
                last_error = exc

        if last_error:
            raise last_error

        return []

    def start_export(self, objects, name, include_dependencies=False):
        export_objects = []

        for obj in objects:
            object_id = first_nonempty(
                obj,
                ["id", "objectId", "assetId"]
            )
            if object_id:
                item = {"id": object_id}
                if include_dependencies:
                    item["includeDependencies"] = True
                export_objects.append(item)

        if not export_objects:
            raise ValueError("No object IDs supplied for export.")

        payload = {
            "name": name,
            "objects": export_objects
        }

        try:
            data = safe_json(
                self.post("/public/core/v3/export", payload)
            )
        except RuntimeError:
            if not include_dependencies:
                raise

            payload["objects"] = [
                {"id": x["id"]}
                for x in export_objects
            ]
            data = safe_json(
                self.post("/public/core/v3/export", payload)
            )

        export_id = first_nonempty(
            data,
            ["id", "jobId", "exportId"]
        )

        if not export_id:
            raise RuntimeError(
                f"Export started but no export job ID returned: {data}"
            )

        return export_id

    def export_status(self, export_id):
        return safe_json(
            self.get(
                f"/public/core/v3/export/{export_id}",
                params={"expand": "objects"}
            )
        )

    @staticmethod
    def _export_state(data):
        if not isinstance(data, dict):
            return "", ""

        status_obj = data.get("status")

        if isinstance(status_obj, dict):
            state = status_obj.get("state") or status_obj.get("status") or ""
            message = status_obj.get("message") or ""
            return str(state).strip().upper(), str(message)

        state = data.get("state") or data.get("jobStatus") or status_obj or ""
        message = data.get("message") or ""
        return str(state).strip().upper(), str(message)

    def wait_for_export(self, export_id):
        deadline = time.time() + EXPORT_TIMEOUT_SECONDS
        previous_display = None
        last_data = None

        success_states = {
            "SUCCESSFUL",
            "SUCCESS",
            "COMPLETED",
            "COMPLETED_SUCCESSFULLY"
        }

        failed_states = {
            "FAILED",
            "FAILURE",
            "ERROR",
            "CANCELLED",
            "CANCELED",
            "ABORTED"
        }

        while time.time() < deadline:
            data = self.export_status(export_id)
            last_data = data

            state, message = self._export_state(data)

            display_state = (state, message)
            if display_state != previous_display:
                print(
                    f"  Export {export_id}: state={state or '<blank>'}"
                    + (f" | {message}" if message else "")
                )
                previous_display = display_state

            if state in success_states:
                return data

            if state in failed_states:
                objects = data.get("objects", []) if isinstance(data, dict) else []
                failed_objects = []

                if isinstance(objects, list):
                    for obj in objects:
                        obj_state, obj_message = self._export_state(obj)
                        if obj_state in failed_states:
                            failed_objects.append({
                                "name": obj.get("name"),
                                "path": obj.get("path"),
                                "state": obj_state,
                                "message": obj_message
                            })

                detail = (
                    f"\nFailed objects: {failed_objects[:20]}"
                    if failed_objects else ""
                )

                raise RuntimeError(
                    f"IICS export {export_id} failed. "
                    f"state={state}, message={message}"
                    f"{detail}"
                )

            time.sleep(EXPORT_POLL_SECONDS)

        state, message = self._export_state(last_data or {})

        raise TimeoutError(
            f"Timed out waiting for export {export_id} after "
            f"{EXPORT_TIMEOUT_SECONDS} seconds. "
            f"Last state={state or '<blank>'}; "
            f"message={message or '<none>'}. "
            "The export job has NOT been cancelled; query the same export ID "
            "again with client.export_status(export_id)."
        )

    def download_export(self, export_id, destination):
        destination = Path(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)

        r = self.get(
            f"/public/core/v3/export/{export_id}/package",
            stream=True
        )

        with destination.open("wb") as fh:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    fh.write(chunk)

        return destination


## 5. Login and retrieve both project inventories

In [ ]:
if not USERNAME or not PASSWORD:
    raise ValueError(
        "Set IICS_USERNAME and IICS_PASSWORD environment variables "
        "or populate USERNAME/PASSWORD in the configuration cell."
    )

client = IICSClient(
    LOGIN_URL,
    USERNAME,
    PASSWORD
)
client.login()

print("Connected to :", client.base_api_url)
print("Organization :", client.org_id)


def relative_asset_path(full_path, project_path):
    fp = normalize_project_path(full_path)
    pp = normalize_project_path(project_path)

    if fp.casefold() == pp.casefold():
        return ""

    prefix = pp + "/"
    if fp.casefold().startswith(prefix.casefold()):
        return fp[len(prefix):]

    return fp


def normalize_inventory(objects, project_path):
    rows = []

    for obj in objects:
        full_path = client.object_path(obj)
        rel_path = relative_asset_path(
            full_path,
            project_path
        )

        rows.append({
            "id": first_nonempty(
                obj,
                ["id", "objectId", "assetId"],
                default=""
            ),
            "name": first_nonempty(
                obj,
                ["name", "objectName", "assetName"],
                default=PurePosixPath(rel_path).name
            ),
            "type": normalize_asset_type(
                first_nonempty(
                    obj,
                    ["type", "objectType", "assetType"],
                    default=""
                )
            ),
            "full_path": full_path,
            "relative_path": rel_path,
            "updated_at": first_nonempty(
                obj,
                [
                    "updateTime",
                    "updatedAt",
                    "lastUpdated",
                    "modifiedTime"
                ],
                default=""
            )
        })

    return pd.DataFrame(rows)


inventory_a = normalize_inventory(
    client.list_objects(PROJECT_A),
    PROJECT_A
)
inventory_b = normalize_inventory(
    client.list_objects(PROJECT_B),
    PROJECT_B
)

print("Project A assets:", len(inventory_a))
print("Project B assets:", len(inventory_b))

display(inventory_a.head(20))
display(inventory_b.head(20))


## 6. Match common assets

In [ ]:
def make_match_key(row):
    asset_type = normalize_asset_type(row["type"])

    if MATCH_BY_RELATIVE_PATH:
        identity = str(
            row["relative_path"]
        ).strip("/").casefold()
    else:
        identity = str(row["name"]).casefold()

    return f"{asset_type}|{identity}"


inventory_a["match_key"] = (
    inventory_a.apply(make_match_key, axis=1)
    if not inventory_a.empty
    else pd.Series(dtype="object")
)
inventory_b["match_key"] = (
    inventory_b.apply(make_match_key, axis=1)
    if not inventory_b.empty
    else pd.Series(dtype="object")
)

a_keys = set(inventory_a["match_key"])
b_keys = set(inventory_b["match_key"])

common_keys = a_keys & b_keys

common_assets = (
    inventory_a[
        inventory_a["match_key"].isin(common_keys)
    ]
    .merge(
        inventory_b[
            inventory_b["match_key"].isin(common_keys)
        ],
        on="match_key",
        suffixes=("_a", "_b")
    )
)

only_a = inventory_a[
    ~inventory_a["match_key"].isin(common_keys)
].copy()

only_b = inventory_b[
    ~inventory_b["match_key"].isin(common_keys)
].copy()

if COMPARE_ASSET_TYPES is None:
    common_to_compare = common_assets.copy()
else:
    allowed = {
        normalize_asset_type(x)
        for x in COMPARE_ASSET_TYPES
    }
    common_to_compare = common_assets[
        common_assets["type_a"].isin(allowed)
    ].copy()

print("Common assets       :", len(common_assets))
print("Only in Project A   :", len(only_a))
print("Only in Project B   :", len(only_b))
print("Selected deep compare:", len(common_to_compare))

display(
    common_to_compare[
        [
            "relative_path_a",
            "type_a",
            "id_a",
            "id_b"
        ]
    ].head(100)
)


## 7. Export matched assets from both projects

In [ ]:
output_root = Path(OUTPUT_DIR)
raw_dir = output_root / "raw_exports"
extract_dir = output_root / "extracted"

raw_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)


def batch_dataframe(df, size):
    for start in range(0, len(df), size):
        yield df.iloc[start:start + size]


def export_side(df, side):
    result_dirs = []

    id_col = f"id_{side.lower()}"

    for batch_no, batch in enumerate(
        batch_dataframe(df, EXPORT_BATCH_SIZE),
        start=1
    ):
        objects = [
            {"id": row[id_col]}
            for _, row in batch.iterrows()
            if row[id_col]
        ]

        if not objects:
            continue

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        job_name = f"compare_{side}_{batch_no}_{stamp}"

        print(
            f"Exporting {side} batch {batch_no}: "
            f"{len(objects)} assets"
        )

        export_id = client.start_export(
            objects,
            name=job_name,
            include_dependencies=INCLUDE_DEPENDENCIES
        )

        client.wait_for_export(export_id)

        zip_path = (
            raw_dir /
            f"{job_name}_{export_id}.zip"
        )
        client.download_export(
            export_id,
            zip_path
        )

        target = (
            extract_dir /
            f"{side}_{batch_no}_{export_id}"
        )
        target.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            zip_path,
            "r"
        ) as zf:
            zf.extractall(target)

        result_dirs.append(target)

        print("  ZIP      :", zip_path)
        print("  Extracted:", target)

    return result_dirs


dirs_a = export_side(
    common_to_compare,
    "A"
)
dirs_b = export_side(
    common_to_compare,
    "B"
)


### Export timeout troubleshooting

The revised polling logic reads the actual IICS export state from `status.state`.

Defaults are now:
- **20 assets per export batch**
- **30-minute timeout per batch**

If an export is still running when the timeout is reached, copy its export ID and inspect that existing job rather than starting a duplicate export.


In [ ]:
# Optional: inspect an existing export job
# EXPORT_ID_TO_CHECK = "paste-export-id-here"
# status_payload = client.export_status(EXPORT_ID_TO_CHECK)
# display(status_payload)
# print("Resolved state:", client._export_state(status_payload))


## 8. Index exported JSON files and associate them with assets

In [ ]:
def load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as fh:
            return json.load(fh)
    except UnicodeDecodeError:
        with open(path, "r", encoding="utf-8-sig") as fh:
            return json.load(fh)
    except Exception:
        return None


def index_json_files(root_dirs):
    rows = []

    for root in root_dirs:
        root = Path(root)

        for path in root.rglob("*.json"):
            obj = load_json(path)
            if obj is None:
                continue

            rows.append({
                "file": str(path),
                "file_name": path.name,
                "relative_file": str(
                    path.relative_to(root)
                ).replace("\\", "/"),
                "size": path.stat().st_size,
                "json": obj,
                "text": compact_json(obj)
            })

    return pd.DataFrame(rows)


json_files_a = index_json_files(dirs_a)
json_files_b = index_json_files(dirs_b)

print("JSON files A:", len(json_files_a))
print("JSON files B:", len(json_files_b))


def score_candidate(
    file_row,
    asset_id,
    asset_name,
    relative_path
):
    text = file_row["text"].casefold()
    filename = file_row["file_name"].casefold()
    relfile = file_row["relative_file"].casefold()

    aid = str(asset_id or "").casefold()
    name = str(asset_name or "").casefold()
    rel = str(relative_path or "").strip("/").casefold()

    score = 0.0

    if aid:
        if aid in text:
            score += 100
        if aid in filename or aid in relfile:
            score += 150

    if name:
        if name in filename:
            score += 80
        if f'"name":"{name}"' in text:
            score += 60
        elif name in text:
            score += 20

    if rel and rel in text:
        score += 40

    score += min(
        file_row["size"] / 100000.0,
        10
    )

    return score


def choose_json(
    files_df,
    asset_id,
    asset_name,
    relative_path
):
    if files_df.empty:
        return None, 0.0

    scored = []

    for idx, row in files_df.iterrows():
        score = score_candidate(
            row,
            asset_id,
            asset_name,
            relative_path
        )
        if score > 0:
            scored.append(
                (score, idx)
            )

    if not scored:
        return None, 0.0

    scored.sort(
        reverse=True
    )
    best_score, best_idx = scored[0]

    return files_df.loc[best_idx], best_score


resolved = []

for _, row in common_to_compare.iterrows():
    a_file, a_score = choose_json(
        json_files_a,
        row["id_a"],
        row["name_a"],
        row["relative_path_a"]
    )
    b_file, b_score = choose_json(
        json_files_b,
        row["id_b"],
        row["name_b"],
        row["relative_path_b"]
    )

    resolved.append({
        "match_key": row["match_key"],
        "relative_path": row["relative_path_a"],
        "asset_type": row["type_a"],
        "asset_name": row["name_a"],
        "id_a": row["id_a"],
        "id_b": row["id_b"],
        "json_file_a": (
            None
            if a_file is None
            else a_file["file"]
        ),
        "json_file_b": (
            None
            if b_file is None
            else b_file["file"]
        ),
        "score_a": a_score,
        "score_b": b_score,
        "json_a": (
            None
            if a_file is None
            else a_file["json"]
        ),
        "json_b": (
            None
            if b_file is None
            else b_file["json"]
        )
    })

asset_json_map = pd.DataFrame(resolved)

display(
    asset_json_map[
        [
            "relative_path",
            "asset_type",
            "json_file_a",
            "json_file_b",
            "score_a",
            "score_b"
        ]
    ]
)


## 9. Normalize volatile IICS metadata

In [ ]:
IGNORE_KEYS = {
    "id",
    "objectid",
    "assetid",
    "uuid",
    "guid",
    "createdby",
    "createdat",
    "createtime",
    "createdtime",
    "updatedby",
    "updatedat",
    "updatetime",
    "modifiedby",
    "modifiedat",
    "lastmodified",
    "lastupdated",
    "revision",
    "revisionid",
    "versionid",
    "etag",
    "checksum",
    "hash",
    "orgid",
    "organizationid",
    "packageid",
    "exportid"
}


def ignore_key(key):
    norm = (
        str(key)
        .replace("_", "")
        .replace("-", "")
        .casefold()
    )

    return (
        norm in IGNORE_KEYS
        or norm.endswith("timestamp")
        or "lastmodified" in norm
    )


def normalize_json(value):
    if isinstance(value, dict):
        result = {}

        for key, child in value.items():
            if ignore_key(key):
                continue
            result[key] = normalize_json(child)

        return {
            key: result[key]
            for key in sorted(
                result,
                key=lambda x: str(x).casefold()
            )
        }

    if isinstance(value, list):
        items = [
            normalize_json(x)
            for x in value
        ]

        preserve_order = any(
            isinstance(x, dict)
            and any(
                str(k).casefold()
                in {
                    "order",
                    "index",
                    "sequence",
                    "position",
                    "ordinal"
                }
                for k in x
            )
            for x in value
        )

        if preserve_order:
            return items

        try:
            return sorted(
                items,
                key=compact_json
            )
        except Exception:
            return items

    if isinstance(value, str):
        return "\n".join(
            line.rstrip()
            for line in value.replace(
                "\r\n",
                "\n"
            ).split("\n")
        ).strip()

    return value


## 10. Semantic parser for transformations, fields, expressions and conditions

In [ ]:
NAME_KEYS = {
    "name",
    "transformationname",
    "objectname",
    "fieldname",
    "portname",
    "parametername",
    "stepname",
    "label"
}

TRANSFORMATION_TYPE_KEYS = {
    "transformationtype",
    "transformtype",
    "transformation",
    "type"
}

TRANSFORMATION_HINTS = {
    "expression",
    "filter",
    "lookup",
    "joiner",
    "aggregator",
    "router",
    "sorter",
    "rank",
    "sequence",
    "source",
    "target",
    "normalizer",
    "union",
    "java",
    "sql",
    "storedprocedure",
    "webservice",
    "hierarchy",
    "datamasking",
    "data masking"
}

EXPRESSION_KEYS = {
    "expression",
    "expr",
    "formula",
    "sql",
    "sqlquery",
    "query"
}

CONDITION_KEYS = {
    "condition",
    "filtercondition",
    "joincondition",
    "lookupcondition",
    "routercondition",
    "conditionexpression",
    "criteria"
}

FIELD_KEYS = {
    "field",
    "fields",
    "port",
    "ports",
    "inputfield",
    "outputfield",
    "sourcefield",
    "targetfield",
    "column",
    "columns",
    "fieldmapping",
    "fieldmappings"
}

PARAMETER_KEYS = {
    "parameter",
    "parameters",
    "parametername",
    "param"
}

CONNECTION_KEYS = {
    "connection",
    "connectionname",
    "runtimeenvironment",
    "runtimeenvironmentname"
}


def recursive_nodes(value, path="$"):
    yield path, value

    if isinstance(value, dict):
        for key, child in value.items():
            yield from recursive_nodes(
                child,
                f"{path}.{key}"
            )

    elif isinstance(value, list):
        for idx, child in enumerate(value):
            yield from recursive_nodes(
                child,
                f"{path}[{idx}]"
            )


def find_name(obj):
    if not isinstance(obj, dict):
        return ""

    for key, value in obj.items():
        if (
            str(key).casefold() in NAME_KEYS
            and isinstance(
                value,
                (str, int, float)
            )
        ):
            return str(value)

    return ""


def find_type(obj):
    if not isinstance(obj, dict):
        return ""

    candidates = []

    for key, value in obj.items():
        kn = (
            str(key)
            .replace("_", "")
            .casefold()
        )

        if (
            kn in TRANSFORMATION_TYPE_KEYS
            and isinstance(
                value,
                (str, int, float)
            )
        ):
            candidates.append(
                str(value)
            )

    for candidate in candidates:
        text = (
            candidate
            .casefold()
            .replace("_", " ")
        )

        if any(
            hint in text
            for hint in TRANSFORMATION_HINTS
        ):
            return candidate

    return (
        candidates[0]
        if candidates
        else ""
    )


def semantic_records(asset_json):
    columns = [
        "category",
        "name",
        "type",
        "property",
        "value",
        "path"
    ]

    if asset_json is None:
        return pd.DataFrame(
            columns=columns
        )

    source = normalize_json(
        asset_json
    )
    rows = []

    exp_keys = {
        x.replace("_", "")
        for x in EXPRESSION_KEYS
    }
    condition_keys = {
        x.replace("_", "")
        for x in CONDITION_KEYS
    }
    field_keys = {
        x.replace("_", "")
        for x in FIELD_KEYS
    }
    parameter_keys = {
        x.replace("_", "")
        for x in PARAMETER_KEYS
    }
    connection_keys = {
        x.replace("_", "")
        for x in CONNECTION_KEYS
    }

    for path, node in recursive_nodes(source):
        if not isinstance(node, dict):
            continue

        name = find_name(node)
        node_type = find_type(node)

        if name and node_type:
            type_text = (
                node_type
                .casefold()
                .replace("_", " ")
            )

            if any(
                hint in type_text
                for hint in TRANSFORMATION_HINTS
            ):
                rows.append({
                    "category": "TRANSFORMATION",
                    "name": name,
                    "type": node_type,
                    "property": "definition",
                    "value": compact_json(node),
                    "path": path
                })

        for key, value in node.items():
            key_norm = (
                str(key)
                .replace("_", "")
                .casefold()
            )

            category = None

            if key_norm in exp_keys:
                category = "EXPRESSION"
            elif key_norm in condition_keys:
                category = "CONDITION"
            elif key_norm in field_keys:
                category = "FIELD_OR_MAPPING"
            elif key_norm in parameter_keys:
                category = "PARAMETER"
            elif key_norm in connection_keys:
                category = "CONNECTION"

            if category is None:
                continue

            if isinstance(
                value,
                (dict, list)
            ):
                value_repr = compact_json(
                    value
                )
            else:
                value_repr = str(value)

            rows.append({
                "category": category,
                "name": (
                    name
                    or PurePosixPath(
                        path.replace(".", "/")
                    ).name
                ),
                "type": node_type,
                "property": str(key),
                "value": value_repr,
                "path": path
            })

    if not rows:
        return pd.DataFrame(
            columns=columns
        )

    return (
        pd.DataFrame(rows)
        .drop_duplicates(
            subset=[
                "category",
                "name",
                "type",
                "property",
                "value"
            ]
        )
        .reset_index(drop=True)
    )


## 11. Compare semantic components and full JSON structure

In [ ]:
def semantic_diff(df_a, df_b):
    key_cols = [
        "category",
        "name",
        "type",
        "property"
    ]

    if df_a.empty and df_b.empty:
        return pd.DataFrame(
            columns=key_cols + [
                "value_a",
                "value_b",
                "change_type"
            ]
        )

    def collapse(df, value_name):
        if df.empty:
            return pd.DataFrame(
                columns=key_cols + [value_name]
            )

        return (
            df.groupby(
                key_cols,
                dropna=False
            )["value"]
            .apply(
                lambda values:
                " || ".join(
                    sorted(
                        set(
                            map(str, values)
                        )
                    )
                )
            )
            .reset_index(
                name=value_name
            )
        )

    a = collapse(
        df_a,
        "value_a"
    )
    b = collapse(
        df_b,
        "value_b"
    )

    merged = a.merge(
        b,
        on=key_cols,
        how="outer",
        indicator=True
    )

    def classify(row):
        if row["_merge"] == "left_only":
            return "ONLY_IN_A"
        if row["_merge"] == "right_only":
            return "ONLY_IN_B"
        if row["value_a"] != row["value_b"]:
            return "CHANGED"
        return "SAME"

    merged["change_type"] = merged.apply(
        classify,
        axis=1
    )

    return (
        merged[
            merged["change_type"] != "SAME"
        ]
        .drop(columns="_merge")
        .reset_index(drop=True)
    )


def flatten_json(value, path="$", result=None):
    if result is None:
        result = {}

    if isinstance(value, dict):
        if not value:
            result[path] = {}

        for key in sorted(
            value,
            key=lambda x: str(x).casefold()
        ):
            flatten_json(
                value[key],
                f"{path}.{key}",
                result
            )

    elif isinstance(value, list):
        if not value:
            result[path] = []

        for idx, child in enumerate(value):
            flatten_json(
                child,
                f"{path}[{idx}]",
                result
            )

    else:
        result[path] = value

    return result


def structural_diff(json_a, json_b):
    a = flatten_json(
        normalize_json(json_a)
    )
    b = flatten_json(
        normalize_json(json_b)
    )

    rows = []

    for path in sorted(
        set(a) | set(b)
    ):
        if path not in b:
            change = "ONLY_IN_A"
        elif path not in a:
            change = "ONLY_IN_B"
        elif a[path] != b[path]:
            change = "CHANGED"
        else:
            continue

        rows.append({
            "json_path": path,
            "change_type": change,
            "value_a": a.get(path),
            "value_b": b.get(path)
        })

    return pd.DataFrame(rows)


summary_rows = []
semantic_frames = []
structural_frames = []

for _, asset in asset_json_map.iterrows():
    json_a = asset["json_a"]
    json_b = asset["json_b"]

    if json_a is None or json_b is None:
        summary_rows.append({
            "relative_path": asset["relative_path"],
            "asset_type": asset["asset_type"],
            "asset_name": asset["asset_name"],
            "comparison_status": "JSON_NOT_FOUND",
            "semantic_differences": None,
            "structural_differences": None
        })
        continue

    sem_a = semantic_records(
        json_a
    )
    sem_b = semantic_records(
        json_b
    )

    sem = semantic_diff(
        sem_a,
        sem_b
    )

    raw = structural_diff(
        json_a,
        json_b
    )

    if not sem.empty:
        sem.insert(
            0,
            "relative_path",
            asset["relative_path"]
        )
        sem.insert(
            1,
            "asset_type",
            asset["asset_type"]
        )
        semantic_frames.append(
            sem
        )

    if not raw.empty:
        raw.insert(
            0,
            "relative_path",
            asset["relative_path"]
        )
        raw.insert(
            1,
            "asset_type",
            asset["asset_type"]
        )
        structural_frames.append(
            raw
        )

    identical = (
        compact_json(
            normalize_json(json_a)
        )
        ==
        compact_json(
            normalize_json(json_b)
        )
    )

    summary_rows.append({
        "relative_path": asset["relative_path"],
        "asset_type": asset["asset_type"],
        "asset_name": asset["asset_name"],
        "comparison_status": (
            "IDENTICAL"
            if identical
            else "DIFFERENT"
        ),
        "semantic_differences": len(sem),
        "structural_differences": len(raw)
    })


asset_comparison_summary = pd.DataFrame(
    summary_rows
)

semantic_differences = (
    pd.concat(
        semantic_frames,
        ignore_index=True
    )
    if semantic_frames
    else pd.DataFrame()
)

structural_differences = (
    pd.concat(
        structural_frames,
        ignore_index=True
    )
    if structural_frames
    else pd.DataFrame()
)

if semantic_differences.empty:
    transformation_differences = pd.DataFrame()
    field_mapping_differences = pd.DataFrame()
else:
    transformation_differences = (
        semantic_differences[
            semantic_differences["category"]
            == "TRANSFORMATION"
        ].copy()
    )

    field_mapping_differences = (
        semantic_differences[
            semantic_differences["category"].isin(
                [
                    "FIELD_OR_MAPPING",
                    "EXPRESSION",
                    "CONDITION"
                ]
            )
        ].copy()
    )

display(
    asset_comparison_summary
)

print(
    "Transformation differences:",
    len(transformation_differences)
)
print(
    "Field/expression/condition differences:",
    len(field_mapping_differences)
)


## 12. Quick summary and missing transformation / mapping views

In [ ]:
print("=" * 90)
print("IICS PROJECT COMPARISON")
print("=" * 90)

print("Project A                    :", PROJECT_A)
print("Project B                    :", PROJECT_B)
print()
print(
    f"Assets in Project A          : {len(inventory_a):,}"
)
print(
    f"Assets in Project B          : {len(inventory_b):,}"
)
print(
    f"Common assets                : {len(common_assets):,}"
)
print(
    f"Only in Project A            : {len(only_a):,}"
)
print(
    f"Only in Project B            : {len(only_b):,}"
)
print(
    f"Deep-compared common assets  : {len(asset_comparison_summary):,}"
)

if not asset_comparison_summary.empty:
    print(
        "Identical assets             :",
        int(
            (
                asset_comparison_summary[
                    "comparison_status"
                ]
                == "IDENTICAL"
            ).sum()
        )
    )
    print(
        "Different assets             :",
        int(
            (
                asset_comparison_summary[
                    "comparison_status"
                ]
                == "DIFFERENT"
            ).sum()
        )
    )
    print(
        "JSON not automatically found :",
        int(
            (
                asset_comparison_summary[
                    "comparison_status"
                ]
                == "JSON_NOT_FOUND"
            ).sum()
        )
    )

if not semantic_differences.empty:
    print("\nSemantic difference summary")
    display(
        semantic_differences
        .groupby(
            [
                "category",
                "change_type"
            ],
            dropna=False
        )
        .size()
        .reset_index(
            name="count"
        )
        .sort_values(
            "count",
            ascending=False
        )
    )

print("\nTRANSFORMATION DIFFERENCES")
display(
    transformation_differences.head(300)
)

print("\nFIELD / MAPPING / EXPRESSION / CONDITION DIFFERENCES")
display(
    field_mapping_differences.head(500)
)


## 13. Export CSV and Excel reports

In [ ]:
report_dir = (
    Path(OUTPUT_DIR)
    / "reports"
)
report_dir.mkdir(
    parents=True,
    exist_ok=True
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

reports = {
    "inventory_project_a": inventory_a,
    "inventory_project_b": inventory_b,
    "common_assets": common_assets,
    "only_project_a": only_a,
    "only_project_b": only_b,
    "comparison_summary": asset_comparison_summary,
    "transformation_diff": transformation_differences,
    "field_mapping_diff": field_mapping_differences,
    "semantic_diff": semantic_differences,
    "structural_diff": structural_differences,
    "json_resolution": asset_json_map.drop(
        columns=[
            "json_a",
            "json_b"
        ],
        errors="ignore"
    )
}

for name, df in reports.items():
    df.to_csv(
        report_dir /
        f"{name}_{timestamp}.csv",
        index=False
    )

excel_path = (
    report_dir /
    f"iics_project_comparison_{timestamp}.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:
    for name, df in reports.items():
        df.to_excel(
            writer,
            sheet_name=name[:31],
            index=False
        )

print("Reports folder:", report_dir)
print("Excel report  :", excel_path)


## 14. Optional: inspect one asset

Set `ASSET_TO_INSPECT` to a relative path shown in the summary, for example:

```python
ASSET_TO_INSPECT = "Mappings/m_Claims_Load"
```


In [ ]:
ASSET_TO_INSPECT = None

if ASSET_TO_INSPECT:
    if not semantic_differences.empty:
        print("Semantic differences")
        display(
            semantic_differences[
                semantic_differences[
                    "relative_path"
                ].str.casefold()
                ==
                ASSET_TO_INSPECT.casefold()
            ]
        )

    if not structural_differences.empty:
        print("Structural differences")
        display(
            structural_differences[
                structural_differences[
                    "relative_path"
                ].str.casefold()
                ==
                ASSET_TO_INSPECT.casefold()
            ]
        )


## Interpretation

The most useful reports are:

- **comparison_summary** — one row per common asset.
- **transformation_diff** — missing, extra, or changed transformations.
- **field_mapping_diff** — field/port mappings, expressions, filters, joins, lookup conditions, and similar mapping logic.
- **only_project_a / only_project_b** — assets missing from one project.
- **structural_diff** — exhaustive normalized JSON differences when a semantic property is not recognized.

`ONLY_IN_A` means the component exists in Project A but not Project B.  
`ONLY_IN_B` means the component exists in Project B but not Project A.  
`CHANGED` means the logical component is present in both but its definition differs.
